## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [ ]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss 
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2" #modelul de embeddinhs

c:\Users\valen\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [6]:
MY_BUBBLE_FILE = "anti_sistem.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: anti_sistem
Texte: 50


,id,agent,text
0,yt_joXkZDqGZQU_Ugyqb1XZ7P8GTnJS_4p4AaABAg,Anti-sistem,Semneaza Bo$$ ca la urmatoarele alegerii nu ma...
1,yt_pO0JOPX1I7Q_UgyExKkzQjU2eOVTG7J4AaABAg,Anti-sistem,ESTE NEVOIE DE O FESTAÑIE LA TOATE NIVELURILE ...
2,yt_6_Hc2S02Duw_Ugz2UatUIFNpL1SP7u54AaABAg,Anti-sistem,Orcii fac ore suplimentare!! Bravo! Daca ati m...
3,yt_YFbJhBc_9jo_Ugx9GFXKkTHUqa4NYcZ4AaABAg,Anti-sistem,Câtă nesimțire!!! Câtă hoție pe fațaă!!! Ce ră...
4,yt_YFbJhBc_9jo_UgzrK_gcZVEbQXhfL614AaABAg,Anti-sistem,"Biserica, aceasta sinecura de sifonat bani, es..."


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [7]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Semneaza Bo$$ ca la urmatoarele alegerii nu mai iesi presedinte. Noi ca tara si popor suntem rupti in cur cu salarii de vietnam si preturi de SIngapore.... dar ajutam cu banii Ukraina ... alta tara corupta la fel si Rusia


In [11]:
metadata[0]

{'id': 'yt_joXkZDqGZQU_Ugyqb1XZ7P8GTnJS_4p4AaABAg',
 'text': 'Semneaza Bo$$ ca la urmatoarele alegerii nu mai iesi presedinte. Noi ca tara si popor suntem rupti in cur cu salarii de vietnam si preturi de SIngapore.... dar ajutam cu banii Ukraina ... alta tara corupta la fel si Rusia',
 'source_channel': 'NicusorDanRO',
 'channel_family': 'mainstream_actor',
 'video_title': '🟢 Declarații de presă comune cu Președintele Ucrainei, Volodîmîr Zelenski, la Palatul Cotroceni',
 'target_refined': 'nicusor_dan',
 'stance_to_target': 'anti',
 'confidence': 0.9,
 'discourse_type': 'T2_grievance_anti_sistem',
 'discourse_subtype': 'grievance_mobilizator',
 'type_confidence': 'medium',
 'agent': 'Anti-sistem',
 'slug': 'anti_sistem',
 'personality': 'furios, suspicios, dezamăgit',
 'speaks': 'acuzator, moralizator, direct',
 'definition': 'vede instituțiile și „sistemul” ca profund compromise'}

## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [8]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

c:\Users\valen\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\valen\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 2/2 [00:00<00:00,  2.05it/s]

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [ ]:
# TODO student:
# Bula mea are 50 texte.
# Au fost generați 50 vectori.
# A doua valoare din embeddings.shape reprezintă 384 de dimensiuni ale fiecărui vector creat,
#   respectiv câte numere sunt folosite pentru a descrie sensul unui text/comentariu.

In [12]:
print("Shape embeddings:" , embeddings.shape)
print(embeddings[0])

Shape embeddings: (50, 384)
[-6.66984096e-02  2.88976170e-02 -3.10920980e-02 -1.08663260e-03
  7.70366937e-02  1.74193690e-03  2.14590821e-02  7.86121748e-03
  9.06166285e-02  9.08237416e-03  2.06962395e-02  1.43609839e-04
  4.29046340e-02 -8.07620492e-03 -1.18060084e-02 -1.12794107e-02
 -4.11439650e-02 -2.51787482e-03  4.55080438e-03  3.63809727e-02
 -3.82861756e-02 -1.78404618e-03  8.01366847e-03  4.67916438e-03
  1.69644833e-01  1.19782100e-02  4.81386632e-02  9.18786041e-04
 -3.87303047e-02 -3.06612793e-02  5.08889072e-02 -4.31232043e-02
  1.46236503e-02  1.17349252e-03  7.01816529e-02 -5.00205196e-02
  8.59272555e-02 -1.16940856e-03 -1.31311722e-03  7.51786679e-02
 -3.37647945e-02 -4.48594689e-02 -9.81308222e-02  2.05468647e-02
  4.98350449e-02 -3.02963648e-02  3.83592397e-02  1.50404824e-02
 -4.49307933e-02  2.77831238e-02  8.33358467e-02  6.25072494e-02
 -1.46316156e-01  6.00081459e-02  7.04245875e-03 -6.28954023e-02
  9.39808860e-02  1.62577964e-02 -7.89162219e-02 -1.64100714e-

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [9]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\anti_sistem
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [ ]:
# TODO student:
# index.faiss există: da
# index.pkl există: da
# index.ntotal este egal cu numărul de texte: da, 50

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [22]:
# Text nou introdus în aplicație

input_text = "Deschiderea fabricilor de armament israeliene în România înseamnă că ne vând țara străinilor și ne bagă în războaie care nu sunt ale noastre."

In [23]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [11]:
# query_vector

In [24]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.496
Text: Wow ! "Ii asigur pe Romani ca nu au motive de ingrijorare. Tara LOR este o tara sigura" Asta nu se considera roman. O fi evreu care ne baga in rahat pentru israel Trebuie suspendat. Nu avem ce ajuta SUA sa atace alte state suverane.

Rezultat 2
Scor: 0.49
Text: psd și aur, împreună cu șefii magistraților, frânează sistematic România, și chiar o sugrumă, pentru a se fura în continuare ! Trebuie o schimbare, nu o stagnare...

Rezultat 3
Scor: 0.472
Text: Jigodiile mafioase ale sistemului dictatorial sint invitati sa nu mai latre minciuni la televiziunile corupte. Inchideti tele manipularea . Dezinformeaza populația. Autorul genocidului , al acțiunilor criminale organizate in plan international sint Netanyahu/ israel assasins la populația civilă, de copii si femei fără apărare , de obiective care nu sint militare.

Rezultat 4
Scor: 0.469
Text: Am impresia că Digi sunt mâhniți că regimul terorist din Iran este distrus...Se vede o stare de nemulțumire că și noi

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

4 dintre comentariile selectate pentru context captează esența agentului anti-sistem, dar nu aduc plus-valoare pentru subiectul specific. Rezultatul 5 este in afara subiectului - vorbește despre probleme interne, impozite, poluare, fără legătură cu tema geopolitică sau anti-imperialistă. Agentul are o componentă anti-israeliană, nu doar anti-sistem intern. Exista riscul sa fie rasist.